In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scipy as sp
import sklearn as sk
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from xgboost import XGBRegressor
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline

def score_dataset(X_train, X_valid, y_train, y_valid):
    model = RandomForestRegressor(n_estimators=100, random_state=0)
    model.fit(X_train, y_train)
    preds = model.predict(X_valid)
    return mean_absolute_error(y_valid, preds)

def score_model(model, X_train, X_valid, y_train, y_valid):
    model.fit(X_train, y_train)
    preds = model.predict(X_valid)
    return mean_absolute_error(y_valid, preds)


path = "../../data/playground-series-s4e1/"



In [3]:

data = pd.read_csv(path + "train.csv")

data.head()


,id,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,0,15674932,Okwudilichukwu,668,France,Male,33.0,3,0.00,2,1.0,0.0,181449.97,0
1,1,15749177,Okwudiliolisa,627,France,Male,33.0,1,0.00,2,1.0,1.0,49503.50,0
2,2,15694510,Hsueh,678,France,Male,40.0,10,0.00,2,1.0,0.0,184866.69,0
3,3,15741417,Kao,581,France,Male,34.0,2,148882.54,1,1.0,1.0,84560.88,0
4,4,15766172,Chiemenam,716,Spain,Male,33.0,5,0.00,2,1.0,1.0,15068.83,0


In [4]:
test_data = pd.read_csv(path + "test.csv")
test_data.head()

,id,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,165034,15773898,Lucchese,586,France,Female,23.0,2,0.00,2,0.0,1.0,160976.75
1,165035,15782418,Nott,683,France,Female,46.0,2,0.00,1,1.0,0.0,72549.27
2,165036,15807120,K?,656,France,Female,34.0,7,0.00,2,1.0,0.0,138882.09
3,165037,15808905,O'Donnell,681,France,Male,36.0,8,0.00,1,1.0,0.0,113931.57
4,165038,15607314,Higgins,752,Germany,Male,38.0,10,121263.62,1,1.0,0.0,139431.00


In [5]:

X_test = test_data.drop(["CustomerId", "Surname"], axis=1)

y_train = data["Exited"]
X_train = data.drop(["CustomerId", "Surname", "Exited"], axis=1)
X_train.head()


,id,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,0,668,France,Male,33.0,3,0.00,2,1.0,0.0,181449.97
1,1,627,France,Male,33.0,1,0.00,2,1.0,1.0,49503.50
2,2,678,France,Male,40.0,10,0.00,2,1.0,0.0,184866.69
3,3,581,France,Male,34.0,2,148882.54,1,1.0,1.0,84560.88
4,4,716,Spain,Male,33.0,5,0.00,2,1.0,1.0,15068.83


In [7]:

grouped = X_train.drop("Gender",axis=1).groupby("Geography")
grouped.mean()



,id,CreditScore,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
Geography,,,,,,,,,
France,82318.367489,656.060638,37.615119,5.027554,37352.477370,1.584387,0.759868,0.505514,112483.924571
Germany,82565.275819,656.920274,39.729209,4.978125,121235.738547,1.445010,0.749783,0.462405,113873.187463
Spain,82985.368045,657.033524,37.922579,5.041974,39795.734545,1.581173,0.742551,0.511419,111570.563508


In [8]:
test_grouped = X_test.drop("Gender", axis = 1).groupby("Geography")
test_grouped.mean()

,id,CreditScore,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
Geography,,,,,,,,,
France,219939.493961,656.037770,37.674144,5.010448,37349.437462,1.585712,0.757784,0.504440,111822.998574
Germany,220192.069445,657.322638,39.634596,4.944025,121348.337856,1.436723,0.749815,0.459392,113946.626038
Spain,220182.636265,657.073600,37.855058,5.010622,39516.542718,1.579601,0.743612,0.505290,112050.232300


In [9]:

print(X_train.shape)
print(X_train.columns)

# for i,x in X_train.iterrows():
#     print(x)

(165034, 11)
Index(['id', 'CreditScore', 'Geography', 'Gender', 'Age', 'Tenure', 'Balance',
       'NumOfProducts', 'HasCrCard', 'IsActiveMember', 'EstimatedSalary'],
      dtype='object')


In [10]:
encoder = OneHotEncoder(sparse_output=False)

encoded_data = encoder.fit_transform(X_train[['Geography']])

# Create DataFrame from encoded data
encoded_df = pd.DataFrame(encoded_data, columns=encoder.get_feature_names_out(['Geography']))

# Concatenate with original DataFrame (optional)
final_df = pd.concat([X_train, encoded_df], axis=1).drop('Geography', axis=1)

# Final DataFrame
final_df["Gender"] = final_df["Gender"] == "Male"
final_df

,id,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_France,Geography_Germany,Geography_Spain
0,0,668,True,33.0,3,0.00,2,1.0,0.0,181449.97,1.0,0.0,0.0
1,1,627,True,33.0,1,0.00,2,1.0,1.0,49503.50,1.0,0.0,0.0
2,2,678,True,40.0,10,0.00,2,1.0,0.0,184866.69,1.0,0.0,0.0
3,3,581,True,34.0,2,148882.54,1,1.0,1.0,84560.88,1.0,0.0,0.0
4,4,716,True,33.0,5,0.00,2,1.0,1.0,15068.83,0.0,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
165029,165029,667,False,33.0,2,0.00,1,1.0,1.0,131834.75,0.0,0.0,1.0
165030,165030,792,True,35.0,3,0.00,1,0.0,0.0,131834.45,1.0,0.0,0.0
165031,165031,565,True,31.0,5,0.00,1,1.0,1.0,127429.56,1.0,0.0,0.0
165032,165032,554,False,30.0,7,161533.00,1,0.0,1.0,71173.03,0.0,0.0,1.0


In [11]:
from sklearn.svm import SVC

model = SVC()

model.fit(X_train,y_train)

print(model.predict(X_test))

ValueError: could not convert string to float: 'France'